In [1]:
# =========================================================
# AI Tour Agent (Telegram bot) — aiogram 3.x — Google Colab
# One-cell запуск. Токены берём ТОЛЬКО из Colab Secrets.
#
# Требуемые Secrets (Colab -> "Secrets"):
#   TELEGRAM_BOT_TOKEN
#   OPENAI_API_KEY
#
# Исправления:
# 1) В ветке "Есть идея" после ответа консультации больше НЕ слетает на старт-меню,
#    и у каждого сообщения в этой ветке есть кнопка 🔊 Озвучить (включая ответы консультации).
# 2) В ветке "Подобрать с нуля" исправлена пагинация направлений (TypeError generator),
#    дальше выводятся туры корректно.
# =========================================================

!pip -q install "aiogram>=3.4,<4.0" "openai>=1.40.0" pandas requests nest_asyncio

import re
import os
import time
import math
import asyncio
import tempfile
import random
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
import requests
import nest_asyncio

from aiogram import Bot, Dispatcher, Router, F
from aiogram.types import (
    Message,
    CallbackQuery,
    ReplyKeyboardMarkup,
    KeyboardButton,
    InlineKeyboardButton,
)
from aiogram.utils.keyboard import InlineKeyboardBuilder
from aiogram.filters import Command, CommandStart
from aiogram.fsm.storage.memory import MemoryStorage
from aiogram.fsm.state import State, StatesGroup
from aiogram.fsm.context import FSMContext

from openai import OpenAI

nest_asyncio.apply()

# -------------------------
# 1) Secrets (ONLY Colab)
# -------------------------
try:
    from google.colab import userdata  # type: ignore
except Exception as e:
    raise RuntimeError("Этот код рассчитан на Google Colab (нужен google.colab.userdata).") from e

TELEGRAM_BOT_TOKEN = userdata.get("TUR_AGENT")
OPENAI_API_KEY = userdata.get("UI_OPENAI_KEY")

if not TELEGRAM_BOT_TOKEN:
    raise ValueError("Не найден секрет TELEGRAM_BOT_TOKEN в Colab Secrets.")
if not OPENAI_API_KEY:
    raise ValueError("Не найден секрет OPENAI_API_KEY в Colab Secrets.")

# -------------------------
# 2) Config
# -------------------------
SHEET_URL = "https://docs.google.com/spreadsheets/d/1aypI3sQSexD6MP4kawiRN7pWWI3iL7frWPHNlmHy12s/edit?usp=sharing"
SHEET_ID = re.search(r"/spreadsheets/d/([a-zA-Z0-9-_]+)", SHEET_URL).group(1)
SHEET_CSV_URL = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/gviz/tq?tqx=out:csv"

OPENAI_TTS_MODEL = "gpt-4o-mini-tts"
OPENAI_TTS_VOICE = "coral"
OPENAI_CHAT_MODEL = "gpt-5"

oai = OpenAI(api_key=OPENAI_API_KEY)

# -------------------------
# 3) Google Sheets helpers
# -------------------------
_SHEET_CACHE: Dict[str, Any] = {"ts": 0, "df": None}

def _norm_col(s: str) -> str:
    return re.sub(r"\s+", " ", str(s)).strip().lower()

def _extract_drive_file_id(url: str) -> Optional[str]:
    if not url:
        return None
    m = re.search(r"/d/([a-zA-Z0-9_-]+)", url)
    if m:
        return m.group(1)
    m = re.search(r"[?&]id=([a-zA-Z0-9_-]+)", url)
    if m:
        return m.group(1)
    return None

def _to_direct_url(url: str) -> str:
    if not url:
        return url
    fid = _extract_drive_file_id(url)
    if fid:
        return f"https://drive.google.com/uc?export=download&id={fid}"
    return url

def load_sheet_df(force: bool = False, ttl_sec: int = 300) -> pd.DataFrame:
    now = time.time()
    if not force and _SHEET_CACHE["df"] is not None and (now - _SHEET_CACHE["ts"]) < ttl_sec:
        return _SHEET_CACHE["df"]

    resp = requests.get(SHEET_CSV_URL, timeout=25)
    resp.raise_for_status()
    df = pd.read_csv(pd.io.common.StringIO(resp.text))

    cols = {_norm_col(c): c for c in df.columns}
    country_col = cols.get("страна") or next((cols[k] for k in cols if "страна" in k or "country" in k), None)
    url_col = cols.get("url") or next((cols[k] for k in cols if k == "url" or "url" in k or "ссылка" in k or "карт" in k), None)
    desc_col = cols.get("описание тура") or next((cols[k] for k in cols if "описание" in k), None)

    if not country_col or not url_col:
        raise ValueError(
            "В таблице не найдены обязательные колонки 'страна' и 'URL'. "
            f"Найденные колонки: {list(df.columns)}"
        )

    df = df.rename(columns={country_col: "страна", url_col: "URL"})
    if desc_col:
        df = df.rename(columns={desc_col: "описание тура"})
    else:
        df["описание тура"] = ""

    df["страна"] = df["страна"].astype(str).fillna("").str.strip()
    df["URL"] = df["URL"].astype(str).fillna("").str.strip()
    df["описание тура"] = df["описание тура"].astype(str).fillna("").str.strip()

    _SHEET_CACHE["ts"] = now
    _SHEET_CACHE["df"] = df
    return df

def base_country_name(s: str) -> str:
    s = re.sub(r"\s+", " ", str(s)).strip()
    s = re.sub(r"\s+\d+$", "", s).strip()
    return s

def get_available_directions() -> List[str]:
    df = load_sheet_df()
    bases = sorted({base_country_name(x) for x in df["страна"].tolist() if str(x).strip()})
    return bases

def get_tours_for_direction(direction_base: str) -> List[Dict[str, str]]:
    df = load_sheet_df()
    rows = df.to_dict(orient="records")
    tours: List[Dict[str, str]] = []
    for r in rows:
        c = base_country_name(r.get("страна", ""))
        if c.lower() == direction_base.lower():
            tours.append({
                "страна": str(r.get("страна", "")).strip(),
                "URL": _to_direct_url(str(r.get("URL", "")).strip()),
                "описание тура": str(r.get("описание тура", "")).strip(),
            })
    return tours

# -------------------------
# 4) Similarity (simple)
# -------------------------
_RU_WORD = re.compile(r"[a-zA-Zа-яА-ЯёЁ0-9]+")

def tokenize(text: str) -> List[str]:
    return _RU_WORD.findall((text or "").lower())

def jaccard_score(a: str, b: str) -> float:
    A = set(tokenize(a))
    B = set(tokenize(b))
    if not A or not B:
        return 0.0
    return len(A & B) / (len(A | B) + 1e-9)

def build_zero_profile_text(data: Dict[str, Any]) -> str:
    parts = []
    parts.append(f"взрослые: {data.get('adults')}")
    parts.append(f"дети: {data.get('kids')}")
    if data.get("kids_ages"):
        parts.append(f"возраст детей: {data.get('kids_ages')}")
    parts.append(f"звезды: {data.get('stars')}")
    parts.append(f"даты: {data.get('dates')}")
    parts.append(f"бюджет: {data.get('budget')}")
    parts.append(f"тип отдыха: {data.get('rest_type')}")
    return ", ".join([p for p in parts if p and "None" not in str(p)])

def rank_tours(profile_text: str, tours: List[Dict[str, str]]) -> List[Tuple[int, float]]:
    scored: List[Tuple[int, float]] = []
    for i, t in enumerate(tours):
        score = jaccard_score(profile_text, t.get("описание тура", "") or "")
        scored.append((i, score))
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored

# -------------------------
# 5) OpenAI helpers (TTS + web_search)
# -------------------------
async def openai_tts_mp3_bytes(text: str) -> bytes:
    text = (text or "").strip()[:4096] or " "
    with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as tmp:
        tmp_path = tmp.name
    try:
        with oai.audio.speech.with_streaming_response.create(
            model=OPENAI_TTS_MODEL,
            voice=OPENAI_TTS_VOICE,
            input=text,
            instructions="Говори чётко, дружелюбно. Это голос ИИ."
        ) as response:
            response.stream_to_file(tmp_path)
        with open(tmp_path, "rb") as f:
            return f.read()
    finally:
        try:
            os.remove(tmp_path)
        except Exception:
            pass

def looks_like_travel_question(text: str) -> bool:
    t = (text or "").lower()
    if len(t) < 3:
        return False
    kw = [
        "виза","паспорт","перелет","самолет","аэропорт","багаж","погода","сезон","когда лучше","отель",
        "звезды","страховк","прививк","маршрут","что посмотреть","достопримечатель","цена","стоимость",
        "бюджет","трансфер","экскурс","тур","поездк","пляж","горы","сафари","круиз","билеты","валюта",
        "безопасн","сим-карт","связь","интернет","розетка","время","еда","включено","входит"
    ]
    return any(k in t for k in kw) or "?" in t

async def openai_answer_with_web_search(profile: str, question: str) -> str:
    prompt = (
        "Ты — туристический консультант. Отвечай кратко, практично, по делу, на русском.\n"
        "Если данных недостаточно — задай 1 уточняющий вопрос в конце.\n\n"
        f"Анкета клиента: {profile}\n"
        f"Вопрос клиента: {question}\n\n"
        "Используй web_search только для актуальной информации. "
        "Не придумывай факты. Если есть ссылки/источники — оставь их в ответе."
    )
    resp = oai.responses.create(
        model=OPENAI_CHAT_MODEL,
        tools=[{"type": "web_search"}],
        input=prompt,
        reasoning={"effort": "low"},
    )
    return (resp.output_text or "").strip() or "Не смог сформировать ответ. Попробуйте переформулировать вопрос."

# -------------------------
# 6) Keyboards
# -------------------------
KB_START = ReplyKeyboardMarkup(
    keyboard=[
        [KeyboardButton(text="У меня есть идея")],
        [KeyboardButton(text="Подобрать с нуля")],
    ],
    resize_keyboard=True,
    selective=True,
)

KB_GLOBAL = ReplyKeyboardMarkup(
    keyboard=[[KeyboardButton(text="Назад"), KeyboardButton(text="Вернуться в начало")]],
    resize_keyboard=True,
    selective=True,
)

KB_CONSULT = ReplyKeyboardMarkup(
    keyboard=[
        [KeyboardButton(text="Нет вопросов")],
        [KeyboardButton(text="Назад"), KeyboardButton(text="Вернуться в начало")],
    ],
    resize_keyboard=True,
    selective=True,
)

def kb_contact_request() -> ReplyKeyboardMarkup:
    return ReplyKeyboardMarkup(
        keyboard=[
            [KeyboardButton(text="📱 Отправить номер", request_contact=True)],
            [KeyboardButton(text="Назад"), KeyboardButton(text="Вернуться в начало")],
        ],
        resize_keyboard=True,
        selective=True,
    )

# -------------------------
# 7) FSM States
# -------------------------
class IdeaFlow(StatesGroup):
    direction = State()
    adults = State()
    adults_manual = State()
    kids = State()
    kids_manual = State()
    kids_ages = State()
    budget = State()
    budget_manual = State()
    rest_type = State()
    rest_type_manual = State()
    consult = State()
    contact_phone = State()
    contact_name = State()

class ZeroFlow(StatesGroup):
    adults = State()
    adults_manual = State()
    kids = State()
    kids_manual = State()
    kids_ages = State()
    stars = State()
    stars_manual = State()
    dates_mode = State()
    dates_manual = State()
    budget = State()
    budget_manual = State()
    rest_type = State()
    rest_type_manual = State()
    choose_direction = State()
    show_tours = State()
    summary = State()
    edit_field = State()
    contact_phone = State()
    contact_name = State()

# -------------------------
# 8) Back stack helpers
# -------------------------
async def push_state(ctx: FSMContext, state: State):
    data = await ctx.get_data()
    hist = data.get("history", [])
    st = await ctx.get_state()
    if st:
        hist.append(st)
    await ctx.update_data(history=hist)
    await ctx.set_state(state)

async def go_back(ctx: FSMContext) -> Optional[str]:
    data = await ctx.get_data()
    hist = data.get("history", [])
    if not hist:
        return None
    prev = hist.pop()
    await ctx.update_data(history=hist)
    await ctx.set_state(prev)
    return prev

async def reset_to_start(ctx: FSMContext):
    await ctx.clear()

# -------------------------
# 9) TTS Store (idea branch)
# -------------------------
TTS_STORE: Dict[str, Dict[str, Any]] = {}
_TTS_COUNTER = 0

def make_tts_key(chat_id: int) -> str:
    global _TTS_COUNTER
    _TTS_COUNTER += 1
    # short key <= 64 chars
    return f"{chat_id}:{int(time.time()*1000)}:{_TTS_COUNTER}:{random.randint(10,99)}"

def add_tts_button(builder: InlineKeyboardBuilder, tts_key: str) -> InlineKeyboardBuilder:
    builder.row(InlineKeyboardButton(text="🔊 Озвучить", callback_data=f"tts:{tts_key}"))
    return builder

async def send_idea_msg(
    bot: Bot,
    chat_id: int,
    text: str,
    reply_kb: Optional[ReplyKeyboardMarkup] = None,
    inline_builder: Optional[InlineKeyboardBuilder] = None,
):
    # message with inline (tts always), then optionally message that sets reply keyboard
    b = inline_builder or InlineKeyboardBuilder()
    key = make_tts_key(chat_id)
    TTS_STORE[key] = {"chat_id": chat_id, "text": text}
    add_tts_button(b, key)
    await bot.send_message(chat_id, text, reply_markup=b.as_markup())
    if reply_kb is not None:
        # IMPORTANT: not blank -> Telegram reliably shows reply keyboard
        await bot.send_message(chat_id, "Выберите действие:", reply_markup=reply_kb)

# -------------------------
# 10) Inline keyboards builders
# -------------------------
def adults_inline(prefix: str) -> InlineKeyboardBuilder:
    b = InlineKeyboardBuilder()
    for n in [1,2,3,4,5,6,7,8]:
        b.button(text=str(n), callback_data=f"{prefix}:{n}")
    b.adjust(4,4)
    b.row(InlineKeyboardButton(text="Ввести вручную", callback_data=f"{prefix}:manual"))
    return b

def kids_inline(prefix: str) -> InlineKeyboardBuilder:
    b = InlineKeyboardBuilder()
    for n in [0,1,2,3,4,5]:
        b.button(text=str(n), callback_data=f"{prefix}:{n}")
    b.adjust(3,3)
    b.row(InlineKeyboardButton(text="Ввести вручную", callback_data=f"{prefix}:manual"))
    return b

def budget_inline(prefix: str) -> InlineKeyboardBuilder:
    b = InlineKeyboardBuilder()
    b.button(text="До 50к", callback_data=f"{prefix}:До 50к")
    b.button(text="50–100к", callback_data=f"{prefix}:50–100к")
    b.button(text="100–200к", callback_data=f"{prefix}:100–200к")
    b.button(text="200к+", callback_data=f"{prefix}:200к+")
    b.adjust(2,2)
    b.row(InlineKeyboardButton(text="Ввести вручную", callback_data=f"{prefix}:manual"))
    return b

def rest_type_inline(prefix: str) -> InlineKeyboardBuilder:
    b = InlineKeyboardBuilder()
    opts = ["Пляж/релакс","Экскурсии","Активный","Семейный","Природа/горы","Круиз"]
    for o in opts:
        b.button(text=o, callback_data=f"{prefix}:{o}")
    b.adjust(2,2,2)
    b.row(InlineKeyboardButton(text="Ввести вручную", callback_data=f"{prefix}:manual"))
    return b

def stars_inline(prefix: str) -> InlineKeyboardBuilder:
    b = InlineKeyboardBuilder()
    for o in ["Любые","3⭐","4⭐","5⭐"]:
        b.button(text=o, callback_data=f"{prefix}:{o}")
    b.adjust(2,2)
    b.row(InlineKeyboardButton(text="Ввести вручную", callback_data=f"{prefix}:manual"))
    return b

def dates_mode_inline(prefix: str) -> InlineKeyboardBuilder:
    b = InlineKeyboardBuilder()
    b.button(text="Точные", callback_data=f"{prefix}:exact")
    b.button(text="Примерно", callback_data=f"{prefix}:approx")
    b.adjust(2)
    b.row(InlineKeyboardButton(text="Ввести вручную", callback_data=f"{prefix}:manual"))
    return b

def dates_approx_inline(prefix: str) -> InlineKeyboardBuilder:
    b = InlineKeyboardBuilder()
    opts = ["В этом месяце","В следующем месяце","Через 2–3 месяца","Лето","Осень","Зима","Весна"]
    for o in opts:
        b.button(text=o, callback_data=f"{prefix}:{o}")
    b.adjust(2,2,3)
    b.row(InlineKeyboardButton(text="Ввести вручную", callback_data=f"{prefix}:manual"))
    return b

def paginate_buttons(items: List[str], page: int, per_page: int, cb_prefix: str) -> InlineKeyboardBuilder:
    # FIXED: no generator indexing; build nav row explicitly
    b = InlineKeyboardBuilder()
    start = page * per_page
    chunk = items[start:start+per_page]
    for it in chunk:
        b.button(text=it, callback_data=f"{cb_prefix}:pick:{it}")
    b.adjust(2,2,2,2)

    nav_buttons: List[InlineKeyboardButton] = []
    if page > 0:
        nav_buttons.append(InlineKeyboardButton(text="⬅️", callback_data=f"{cb_prefix}:page:{page-1}"))
    if start + per_page < len(items):
        nav_buttons.append(InlineKeyboardButton(text="➡️", callback_data=f"{cb_prefix}:page:{page+1}"))
    if nav_buttons:
        b.row(*nav_buttons)
    return b

# -------------------------
# 11) Utils
# -------------------------
def normalize_phone(s: str) -> Optional[str]:
    digits = re.sub(r"\D+", "", s or "")
    if len(digits) < 10:
        return None
    return digits

def safe_text(s: Any) -> str:
    return re.sub(r"\s+", " ", str(s or "")).strip()

# -------------------------
# 12) Router + commands
# -------------------------
router = Router()

@router.message(Command("help"))
async def cmd_help(m: Message):
    await m.answer(
        "Я AI-тур-агент: помогу собрать заявку, проконсультировать по направлению и подобрать туры из базы.\n\n"
        "Команды:\n"
        "/start — начать заново\n"
        "/help — помощь\n"
        "/contacts — контакты менеджера\n"
        "/site — сайт",
        reply_markup=KB_START
    )

@router.message(Command("contacts"))
async def cmd_contacts(m: Message):
    await m.answer("Менеджер: Алёна\nТелефон: 89998765432")

@router.message(Command("site"))
async def cmd_site(m: Message):
    await m.answer("https://neural-university.ru/")

@router.message(CommandStart())
async def start(m: Message, state: FSMContext):
    await reset_to_start(state)
    await m.answer(
        "Привет! Я AI-тур-агент — помогу собрать заявку. У вас уже есть идея направления или подбираем с нуля?",
        reply_markup=KB_START
    )

@router.message(F.text == "Вернуться в начало")
async def go_home(m: Message, state: FSMContext):
    await reset_to_start(state)
    await m.answer(
        "Привет! Я AI-тур-агент — помогу собрать заявку. У вас уже есть идея направления или подбираем с нуля?",
        reply_markup=KB_START
    )

@router.message(F.text == "Назад")
async def back(m: Message, state: FSMContext, bot: Bot):
    prev = await go_back(state)
    if not prev:
        await m.answer("Вы уже в начале. Нажмите «Вернуться в начало».", reply_markup=KB_GLOBAL)
        return

    st = await state.get_state()
    chat_id = m.chat.id

    # IDEA re-render
    if st == IdeaFlow.direction.state:
        await send_idea_msg(bot, chat_id, "Куда хотите поехать? Напишите направление текстом.", reply_kb=KB_GLOBAL)
    elif st == IdeaFlow.adults.state:
        await send_idea_msg(bot, chat_id, "Сколько взрослых?", reply_kb=KB_GLOBAL, inline_builder=adults_inline("idea_adults"))
    elif st == IdeaFlow.kids.state:
        await send_idea_msg(bot, chat_id, "Сколько детей?", reply_kb=KB_GLOBAL, inline_builder=kids_inline("idea_kids"))
    elif st == IdeaFlow.kids_ages.state:
        data = await state.get_data()
        n = int(data.get("kids", 0))
        await send_idea_msg(bot, chat_id, f"Укажите возраст каждого ребёнка через запятую (всего детей: {n}).", reply_kb=KB_GLOBAL)
    elif st == IdeaFlow.budget.state:
        await send_idea_msg(bot, chat_id, "Бюджет:", reply_kb=KB_GLOBAL, inline_builder=budget_inline("idea_budget"))
    elif st == IdeaFlow.rest_type.state:
        await send_idea_msg(bot, chat_id, "Тип отдыха:", reply_kb=KB_GLOBAL, inline_builder=rest_type_inline("idea_type"))
    elif st == IdeaFlow.consult.state:
        await send_idea_msg(bot, chat_id, "Режим консультации включён. Задайте вопрос по поездке или нажмите «Нет вопросов».", reply_kb=KB_CONSULT)

    # ZERO re-render (reply keyboard only)
    elif st == ZeroFlow.adults.state:
        await m.answer("Сколько взрослых?", reply_markup=KB_GLOBAL)
        await m.answer("Выберите:", reply_markup=adults_inline("zero_adults").as_markup())
    elif st == ZeroFlow.kids.state:
        await m.answer("Сколько детей?", reply_markup=KB_GLOBAL)
        await m.answer("Выберите:", reply_markup=kids_inline("zero_kids").as_markup())
    elif st == ZeroFlow.kids_ages.state:
        data = await state.get_data()
        n = int(data.get("kids", 0))
        await m.answer(f"Укажите возраст каждого ребёнка через запятую (всего детей: {n}).", reply_markup=KB_GLOBAL)
    elif st == ZeroFlow.stars.state:
        await m.answer("Звёзды отеля:", reply_markup=KB_GLOBAL)
        await m.answer("Выберите:", reply_markup=stars_inline("zero_stars").as_markup())
    elif st == ZeroFlow.dates_mode.state:
        await m.answer("Даты поездки:", reply_markup=KB_GLOBAL)
        await m.answer("Выберите:", reply_markup=dates_mode_inline("zero_dates").as_markup())
    elif st == ZeroFlow.budget.state:
        await m.answer("Бюджет:", reply_markup=KB_GLOBAL)
        await m.answer("Выберите:", reply_markup=budget_inline("zero_budget").as_markup())
    elif st == ZeroFlow.rest_type.state:
        await m.answer("Тип отдыха:", reply_markup=KB_GLOBAL)
        await m.answer("Выберите:", reply_markup=rest_type_inline("zero_type").as_markup())
    elif st == ZeroFlow.choose_direction.state:
        data = await state.get_data()
        page = int(data.get("dir_page", 0))
        dirs = get_available_directions()
        await m.answer("Выберите направление из базы:", reply_markup=KB_GLOBAL)
        await m.answer("Направления:", reply_markup=paginate_buttons(dirs, page, 8, "zero_dir").as_markup())
    elif st == ZeroFlow.contact_phone.state:
        await m.answer("Отправьте номер телефона (кнопкой) или напишите вручную:", reply_markup=kb_contact_request())
    elif st == IdeaFlow.contact_phone.state:
        await m.answer("Отправьте номер телефона (кнопкой) или напишите вручную:", reply_markup=kb_contact_request())
    else:
        await m.answer("Ок.", reply_markup=KB_GLOBAL)

# -------------------------
# 13) Branch selection
# -------------------------
@router.message(F.text == "У меня есть идея")
async def choose_idea(m: Message, state: FSMContext, bot: Bot):
    await reset_to_start(state)
    await state.update_data(flow="idea", history=[])
    await push_state(state, IdeaFlow.direction)
    await send_idea_msg(bot, m.chat.id, "Куда хотите поехать? Напишите направление текстом.", reply_kb=KB_GLOBAL)

@router.message(F.text == "Подобрать с нуля")
async def choose_zero(m: Message, state: FSMContext):
    await reset_to_start(state)
    await state.update_data(flow="zero", history=[])
    await push_state(state, ZeroFlow.adults)
    await m.answer("Сколько взрослых?", reply_markup=KB_GLOBAL)
    await m.answer("Выберите:", reply_markup=adults_inline("zero_adults").as_markup())

# -------------------------
# 14) IDEA flow
# -------------------------
@router.message(IdeaFlow.direction)
async def idea_direction(m: Message, state: FSMContext, bot: Bot):
    txt = safe_text(m.text)
    if not txt:
        await send_idea_msg(bot, m.chat.id, "Напишите направление текстом (например: Непал, Абхазия, Сафари).", reply_kb=KB_GLOBAL)
        return
    await state.update_data(direction=txt)
    await push_state(state, IdeaFlow.adults)
    await send_idea_msg(bot, m.chat.id, "Сколько взрослых?", reply_kb=KB_GLOBAL, inline_builder=adults_inline("idea_adults"))

@router.callback_query(F.data.startswith("idea_adults:"))
async def idea_adults_cb(c: CallbackQuery, state: FSMContext, bot: Bot):
    await c.answer()
    val = c.data.split(":", 1)[1]
    if val == "manual":
        await push_state(state, IdeaFlow.adults_manual)
        await send_idea_msg(bot, c.message.chat.id, "Введите число взрослых (1–8):", reply_kb=KB_GLOBAL)
        return
    n = int(val)
    if not (1 <= n <= 8):
        await send_idea_msg(bot, c.message.chat.id, "Взрослые должны быть 1–8. Выберите снова.", reply_kb=KB_GLOBAL, inline_builder=adults_inline("idea_adults"))
        return
    await state.update_data(adults=n)
    await push_state(state, IdeaFlow.kids)
    await send_idea_msg(bot, c.message.chat.id, "Сколько детей?", reply_kb=KB_GLOBAL, inline_builder=kids_inline("idea_kids"))

@router.message(IdeaFlow.adults_manual)
async def idea_adults_manual(m: Message, state: FSMContext, bot: Bot):
    txt = safe_text(m.text)
    if not txt.isdigit():
        await send_idea_msg(bot, m.chat.id, "Нужно число 1–8. Введите ещё раз:", reply_kb=KB_GLOBAL)
        return
    n = int(txt)
    if not (1 <= n <= 8):
        await send_idea_msg(bot, m.chat.id, "Взрослые строго 1–8. Введите ещё раз:", reply_kb=KB_GLOBAL)
        return
    await state.update_data(adults=n)
    await push_state(state, IdeaFlow.kids)
    await send_idea_msg(bot, m.chat.id, "Сколько детей?", reply_kb=KB_GLOBAL, inline_builder=kids_inline("idea_kids"))

@router.callback_query(F.data.startswith("idea_kids:"))
async def idea_kids_cb(c: CallbackQuery, state: FSMContext, bot: Bot):
    await c.answer()
    val = c.data.split(":", 1)[1]
    if val == "manual":
        await push_state(state, IdeaFlow.kids_manual)
        await send_idea_msg(bot, c.message.chat.id, "Введите число детей (0–5):", reply_kb=KB_GLOBAL)
        return
    k = int(val)
    if not (0 <= k <= 5):
        await send_idea_msg(bot, c.message.chat.id, "Дети должны быть 0–5. Выберите снова.", reply_kb=KB_GLOBAL, inline_builder=kids_inline("idea_kids"))
        return
    await state.update_data(kids=k)
    if k > 0:
        await push_state(state, IdeaFlow.kids_ages)
        await send_idea_msg(bot, c.message.chat.id, f"Укажите возраст каждого ребёнка через запятую (всего детей: {k}).", reply_kb=KB_GLOBAL)
    else:
        await push_state(state, IdeaFlow.budget)
        await send_idea_msg(bot, c.message.chat.id, "Бюджет:", reply_kb=KB_GLOBAL, inline_builder=budget_inline("idea_budget"))

@router.message(IdeaFlow.kids_manual)
async def idea_kids_manual(m: Message, state: FSMContext, bot: Bot):
    txt = safe_text(m.text)
    if not txt.isdigit():
        await send_idea_msg(bot, m.chat.id, "Нужно число 0–5. Введите ещё раз:", reply_kb=KB_GLOBAL)
        return
    k = int(txt)
    if not (0 <= k <= 5):
        await send_idea_msg(bot, m.chat.id, "Дети строго 0–5. Введите ещё раз:", reply_kb=KB_GLOBAL)
        return
    await state.update_data(kids=k)
    if k > 0:
        await push_state(state, IdeaFlow.kids_ages)
        await send_idea_msg(bot, m.chat.id, f"Укажите возраст каждого ребёнка через запятую (всего детей: {k}).", reply_kb=KB_GLOBAL)
    else:
        await push_state(state, IdeaFlow.budget)
        await send_idea_msg(bot, m.chat.id, "Бюджет:", reply_kb=KB_GLOBAL, inline_builder=budget_inline("idea_budget"))

@router.message(IdeaFlow.kids_ages)
async def idea_kids_ages(m: Message, state: FSMContext, bot: Bot):
    data = await state.get_data()
    k = int(data.get("kids", 0))
    parts = [p.strip() for p in (m.text or "").split(",") if p.strip()]
    if len(parts) != k:
        await send_idea_msg(bot, m.chat.id, f"Нужно указать ровно {k} возраст(ов). Попробуйте ещё раз:", reply_kb=KB_GLOBAL)
        return
    ages = []
    for p in parts:
        if not re.fullmatch(r"\d{1,2}", p):
            await send_idea_msg(bot, m.chat.id, "Возраст должен быть числом. Введите ещё раз:", reply_kb=KB_GLOBAL)
            return
        ages.append(int(p))
    await state.update_data(kids_ages=ages)
    await push_state(state, IdeaFlow.budget)
    await send_idea_msg(bot, m.chat.id, "Бюджет:", reply_kb=KB_GLOBAL, inline_builder=budget_inline("idea_budget"))

@router.callback_query(F.data.startswith("idea_budget:"))
async def idea_budget_cb(c: CallbackQuery, state: FSMContext, bot: Bot):
    await c.answer()
    val = c.data.split(":", 1)[1]
    if val == "manual":
        await push_state(state, IdeaFlow.budget_manual)
        await send_idea_msg(bot, c.message.chat.id, "Введите бюджет текстом (например: 120 000 на двоих):", reply_kb=KB_GLOBAL)
        return
    await state.update_data(budget=val)
    await push_state(state, IdeaFlow.rest_type)
    await send_idea_msg(bot, c.message.chat.id, "Тип отдыха:", reply_kb=KB_GLOBAL, inline_builder=rest_type_inline("idea_type"))

@router.message(IdeaFlow.budget_manual)
async def idea_budget_manual(m: Message, state: FSMContext, bot: Bot):
    txt = safe_text(m.text)
    if not txt:
        await send_idea_msg(bot, m.chat.id, "Введите бюджет текстом:", reply_kb=KB_GLOBAL)
        return
    await state.update_data(budget=txt)
    await push_state(state, IdeaFlow.rest_type)
    await send_idea_msg(bot, m.chat.id, "Тип отдыха:", reply_kb=KB_GLOBAL, inline_builder=rest_type_inline("idea_type"))

@router.callback_query(F.data.startswith("idea_type:"))
async def idea_type_cb(c: CallbackQuery, state: FSMContext, bot: Bot):
    await c.answer()
    val = c.data.split(":", 1)[1]
    if val == "manual":
        await push_state(state, IdeaFlow.rest_type_manual)
        await send_idea_msg(bot, c.message.chat.id, "Введите тип отдыха текстом:", reply_kb=KB_GLOBAL)
        return
    await state.update_data(rest_type=val)
    await push_state(state, IdeaFlow.consult)
    await send_idea_msg(
        bot,
        c.message.chat.id,
        "Режим консультации включён. Задайте вопрос по поездке или нажмите «Нет вопросов».",
        reply_kb=KB_CONSULT
    )

@router.message(IdeaFlow.rest_type_manual)
async def idea_type_manual(m: Message, state: FSMContext, bot: Bot):
    txt = safe_text(m.text)
    if not txt:
        await send_idea_msg(bot, m.chat.id, "Введите тип отдыха текстом:", reply_kb=KB_GLOBAL)
        return
    await state.update_data(rest_type=txt)
    await push_state(state, IdeaFlow.consult)
    await send_idea_msg(
        bot,
        m.chat.id,
        "Режим консультации включён. Задайте вопрос по поездке или нажмите «Нет вопросов».",
        reply_kb=KB_CONSULT
    )

@router.message(IdeaFlow.consult, F.text == "Нет вопросов")
async def idea_no_questions(m: Message, state: FSMContext, bot: Bot):
    data = await state.get_data()
    ank = [
        "📌 Анкета:",
        f"Направление: {data.get('direction')}",
        f"Состав: взрослые {data.get('adults')}, дети {data.get('kids')}" + (f" (возраст: {data.get('kids_ages')})" if data.get("kids_ages") else ""),
        f"Бюджет: {data.get('budget')}",
        f"Тип отдыха: {data.get('rest_type')}",
        "",
        "Отправьте номер телефона (кнопкой) или напишите вручную:"
    ]
    await push_state(state, IdeaFlow.contact_phone)
    await send_idea_msg(bot, m.chat.id, "\n".join(ank), reply_kb=kb_contact_request())

@router.message(IdeaFlow.consult)
async def idea_consult_question(m: Message, state: FSMContext, bot: Bot):
    txt = safe_text(m.text)
    if txt in {"Назад", "Вернуться в начало"}:
        return
    if not txt:
        await send_idea_msg(bot, m.chat.id, "Напишите вопрос текстом или нажмите «Нет вопросов».", reply_kb=KB_CONSULT)
        return

    if not looks_like_travel_question(txt):
        await send_idea_msg(bot, m.chat.id, "Этот вопрос выходит за пределы консультации по туру/поездке.", reply_kb=KB_CONSULT)
        return

    data = await state.get_data()
    profile = (
        f"направление: {data.get('direction')}, взрослые: {data.get('adults')}, дети: {data.get('kids')}, "
        f"бюджет: {data.get('budget')}, тип отдыха: {data.get('rest_type')}"
    )
    try:
        answer = await openai_answer_with_web_search(profile, txt)
    except Exception as e:
        answer = f"Не получилось выполнить поиск/ответ: {e}"

    # IMPORTANT: keep consult reply keyboard (so it doesn't show start)
    await send_idea_msg(bot, m.chat.id, answer, reply_kb=KB_CONSULT)

@router.message(IdeaFlow.contact_phone)
async def idea_phone(m: Message, state: FSMContext, bot: Bot):
    phone = None
    if m.contact and m.contact.phone_number:
        phone = normalize_phone(m.contact.phone_number)
    else:
        phone = normalize_phone(m.text or "")
    if not phone:
        await send_idea_msg(bot, m.chat.id, "Не распознал телефон. Отправьте кнопкой или напишите вручную (минимум 10 цифр).", reply_kb=kb_contact_request())
        return
    await state.update_data(phone=phone)
    await push_state(state, IdeaFlow.contact_name)
    await send_idea_msg(bot, m.chat.id, "Введите Фамилию Имя:", reply_kb=KB_GLOBAL)

@router.message(IdeaFlow.contact_name)
async def idea_name(m: Message, state: FSMContext, bot: Bot):
    fio = safe_text(m.text)
    if len(fio.split()) < 2:
        await send_idea_msg(bot, m.chat.id, "Нужно в формате «Фамилия Имя». Введите ещё раз:", reply_kb=KB_GLOBAL)
        return
    data = await state.get_data()
    phone = data.get("phone", "")
    await send_idea_msg(
        bot,
        m.chat.id,
        f"Спасибо, {fio}! Мы отправили вашу заявку менеджеру — скоро с вами свяжутся для оплаты.\nТелефон: {phone}",
        reply_kb=KB_START
    )
    await reset_to_start(state)

# -------------------------
# 15) ZERO flow
# -------------------------
@router.callback_query(F.data.startswith("zero_adults:"))
async def zero_adults_cb(c: CallbackQuery, state: FSMContext):
    await c.answer()
    val = c.data.split(":", 1)[1]
    if val == "manual":
        await push_state(state, ZeroFlow.adults_manual)
        await c.message.answer("Введите число взрослых (1–8):", reply_markup=KB_GLOBAL)
        return
    n = int(val)
    if not (1 <= n <= 8):
        await c.message.answer("Взрослые строго 1–8. Выберите снова.", reply_markup=KB_GLOBAL)
        await c.message.answer("Выберите:", reply_markup=adults_inline("zero_adults").as_markup())
        return
    await state.update_data(adults=n)
    await push_state(state, ZeroFlow.kids)
    await c.message.answer("Сколько детей?", reply_markup=KB_GLOBAL)
    await c.message.answer("Выберите:", reply_markup=kids_inline("zero_kids").as_markup())

@router.message(ZeroFlow.adults_manual)
async def zero_adults_manual(m: Message, state: FSMContext):
    txt = safe_text(m.text)
    if not txt.isdigit():
        await m.answer("Нужно число 1–8. Введите ещё раз:", reply_markup=KB_GLOBAL)
        return
    n = int(txt)
    if not (1 <= n <= 8):
        await m.answer("Взрослые строго 1–8. Введите ещё раз:", reply_markup=KB_GLOBAL)
        return
    await state.update_data(adults=n)
    await push_state(state, ZeroFlow.kids)
    await m.answer("Сколько детей?", reply_markup=KB_GLOBAL)
    await m.answer("Выберите:", reply_markup=kids_inline("zero_kids").as_markup())

@router.callback_query(F.data.startswith("zero_kids:"))
async def zero_kids_cb(c: CallbackQuery, state: FSMContext):
    await c.answer()
    val = c.data.split(":", 1)[1]
    if val == "manual":
        await push_state(state, ZeroFlow.kids_manual)
        await c.message.answer("Введите число детей (0–5):", reply_markup=KB_GLOBAL)
        return
    k = int(val)
    if not (0 <= k <= 5):
        await c.message.answer("Дети строго 0–5. Выберите снова.", reply_markup=KB_GLOBAL)
        await c.message.answer("Выберите:", reply_markup=kids_inline("zero_kids").as_markup())
        return
    await state.update_data(kids=k)
    if k > 0:
        await push_state(state, ZeroFlow.kids_ages)
        await c.message.answer(f"Укажите возраст каждого ребёнка через запятую (всего детей: {k}).", reply_markup=KB_GLOBAL)
    else:
        await push_state(state, ZeroFlow.stars)
        await c.message.answer("Звёзды отеля:", reply_markup=KB_GLOBAL)
        await c.message.answer("Выберите:", reply_markup=stars_inline("zero_stars").as_markup())

@router.message(ZeroFlow.kids_manual)
async def zero_kids_manual(m: Message, state: FSMContext):
    txt = safe_text(m.text)
    if not txt.isdigit():
        await m.answer("Нужно число 0–5. Введите ещё раз:", reply_markup=KB_GLOBAL)
        return
    k = int(txt)
    if not (0 <= k <= 5):
        await m.answer("Дети строго 0–5. Введите ещё раз:", reply_markup=KB_GLOBAL)
        return
    await state.update_data(kids=k)
    if k > 0:
        await push_state(state, ZeroFlow.kids_ages)
        await m.answer(f"Укажите возраст каждого ребёнка через запятую (всего детей: {k}).", reply_markup=KB_GLOBAL)
    else:
        await push_state(state, ZeroFlow.stars)
        await m.answer("Звёзды отеля:", reply_markup=KB_GLOBAL)
        await m.answer("Выберите:", reply_markup=stars_inline("zero_stars").as_markup())

@router.message(ZeroFlow.kids_ages)
async def zero_kids_ages(m: Message, state: FSMContext):
    data = await state.get_data()
    k = int(data.get("kids", 0))
    parts = [p.strip() for p in (m.text or "").split(",") if p.strip()]
    if len(parts) != k:
        await m.answer(f"Нужно указать ровно {k} возраст(ов). Попробуйте ещё раз:", reply_markup=KB_GLOBAL)
        return
    ages = []
    for p in parts:
        if not re.fullmatch(r"\d{1,2}", p):
            await m.answer("Возраст должен быть числом. Введите ещё раз:", reply_markup=KB_GLOBAL)
            return
        ages.append(int(p))
    await state.update_data(kids_ages=ages)
    await push_state(state, ZeroFlow.stars)
    await m.answer("Звёзды отеля:", reply_markup=KB_GLOBAL)
    await m.answer("Выберите:", reply_markup=stars_inline("zero_stars").as_markup())

@router.callback_query(F.data.startswith("zero_stars:"))
async def zero_stars_cb(c: CallbackQuery, state: FSMContext):
    await c.answer()
    val = c.data.split(":", 1)[1]
    if val == "manual":
        await push_state(state, ZeroFlow.stars_manual)
        await c.message.answer("Введите звёзды отеля текстом (например: 4⭐ или любые):", reply_markup=KB_GLOBAL)
        return
    await state.update_data(stars=val)
    await push_state(state, ZeroFlow.dates_mode)
    await c.message.answer("Даты поездки:", reply_markup=KB_GLOBAL)
    await c.message.answer("Выберите:", reply_markup=dates_mode_inline("zero_dates").as_markup())

@router.message(ZeroFlow.stars_manual)
async def zero_stars_manual(m: Message, state: FSMContext):
    txt = safe_text(m.text)
    if not txt:
        await m.answer("Введите звёзды отеля:", reply_markup=KB_GLOBAL)
        return
    await state.update_data(stars=txt)
    await push_state(state, ZeroFlow.dates_mode)
    await m.answer("Даты поездки:", reply_markup=KB_GLOBAL)
    await m.answer("Выберите:", reply_markup=dates_mode_inline("zero_dates").as_markup())

@router.callback_query(F.data.startswith("zero_dates:"))
async def zero_dates_cb(c: CallbackQuery, state: FSMContext):
    await c.answer()
    mode = c.data.split(":", 1)[1]
    if mode == "manual":
        await push_state(state, ZeroFlow.dates_manual)
        await c.message.answer("Введите даты текстом (например: 10.01–20.01 или 'конец мая'):", reply_markup=KB_GLOBAL)
        return
    if mode == "exact":
        await push_state(state, ZeroFlow.dates_manual)
        await c.message.answer("Введите точные даты (например: 10.01–20.01 или 10.01.2026–20.01.2026):", reply_markup=KB_GLOBAL)
        return
    if mode == "approx":
        await c.message.answer("Выберите примерный период:", reply_markup=KB_GLOBAL)
        await c.message.answer("Период:", reply_markup=dates_approx_inline("zero_dates_approx").as_markup())
        return

@router.callback_query(F.data.startswith("zero_dates_approx:"))
async def zero_dates_approx_cb(c: CallbackQuery, state: FSMContext):
    await c.answer()
    val = c.data.split(":", 1)[1]
    if val == "manual":
        await push_state(state, ZeroFlow.dates_manual)
        await c.message.answer("Введите период текстом (например: 'июль 2026' или 'весной'):", reply_markup=KB_GLOBAL)
        return
    await state.update_data(dates=val)
    await push_state(state, ZeroFlow.budget)
    await c.message.answer("Бюджет:", reply_markup=KB_GLOBAL)
    await c.message.answer("Выберите:", reply_markup=budget_inline("zero_budget").as_markup())

@router.message(ZeroFlow.dates_manual)
async def zero_dates_manual(m: Message, state: FSMContext):
    txt = safe_text(m.text)
    if not txt:
        await m.answer("Введите даты/период:", reply_markup=KB_GLOBAL)
        return
    await state.update_data(dates=txt)
    await push_state(state, ZeroFlow.budget)
    await m.answer("Бюджет:", reply_markup=KB_GLOBAL)
    await m.answer("Выберите:", reply_markup=budget_inline("zero_budget").as_markup())

@router.callback_query(F.data.startswith("zero_budget:"))
async def zero_budget_cb(c: CallbackQuery, state: FSMContext):
    await c.answer()
    val = c.data.split(":", 1)[1]
    if val == "manual":
        await push_state(state, ZeroFlow.budget_manual)
        await c.message.answer("Введите бюджет текстом (например: 150 000 на двоих):", reply_markup=KB_GLOBAL)
        return
    await state.update_data(budget=val)
    await push_state(state, ZeroFlow.rest_type)
    await c.message.answer("Тип отдыха:", reply_markup=KB_GLOBAL)
    await c.message.answer("Выберите:", reply_markup=rest_type_inline("zero_type").as_markup())

@router.message(ZeroFlow.budget_manual)
async def zero_budget_manual(m: Message, state: FSMContext):
    txt = safe_text(m.text)
    if not txt:
        await m.answer("Введите бюджет текстом:", reply_markup=KB_GLOBAL)
        return
    await state.update_data(budget=txt)
    await push_state(state, ZeroFlow.rest_type)
    await m.answer("Тип отдыха:", reply_markup=KB_GLOBAL)
    await m.answer("Выберите:", reply_markup=rest_type_inline("zero_type").as_markup())

@router.callback_query(F.data.startswith("zero_type:"))
async def zero_type_cb(c: CallbackQuery, state: FSMContext):
    await c.answer()
    val = c.data.split(":", 1)[1]
    if val == "manual":
        await push_state(state, ZeroFlow.rest_type_manual)
        await c.message.answer("Введите тип отдыха текстом:", reply_markup=KB_GLOBAL)
        return

    await state.update_data(rest_type=val)

    # load directions from sheet
    try:
        dirs = get_available_directions()
    except Exception as e:
        await c.message.answer(f"Ошибка чтения Google Sheets: {e}", reply_markup=KB_GLOBAL)
        return

    if not dirs:
        await c.message.answer("В базе пока нет направлений.", reply_markup=KB_GLOBAL)
        return

    await state.update_data(dir_page=0)
    await push_state(state, ZeroFlow.choose_direction)
    await c.message.answer("Выберите направление из доступных в базе:", reply_markup=KB_GLOBAL)
    await c.message.answer("Направления:", reply_markup=paginate_buttons(dirs, 0, 8, "zero_dir").as_markup())

@router.message(ZeroFlow.rest_type_manual)
async def zero_type_manual(m: Message, state: FSMContext):
    txt = safe_text(m.text)
    if not txt:
        await m.answer("Введите тип отдыха текстом:", reply_markup=KB_GLOBAL)
        return
    await state.update_data(rest_type=txt)

    try:
        dirs = get_available_directions()
    except Exception as e:
        await m.answer(f"Ошибка чтения Google Sheets: {e}", reply_markup=KB_GLOBAL)
        return

    if not dirs:
        await m.answer("В базе пока нет направлений.", reply_markup=KB_GLOBAL)
        return

    await state.update_data(dir_page=0)
    await push_state(state, ZeroFlow.choose_direction)
    await m.answer("Выберите направление из доступных в базе:", reply_markup=KB_GLOBAL)
    await m.answer("Направления:", reply_markup=paginate_buttons(dirs, 0, 8, "zero_dir").as_markup())

@router.callback_query(F.data.startswith("zero_dir:page:"))
async def zero_dir_page(c: CallbackQuery, state: FSMContext):
    await c.answer()
    page = int(c.data.split(":")[-1])
    await state.update_data(dir_page=page)
    dirs = get_available_directions()
    await c.message.edit_reply_markup(reply_markup=paginate_buttons(dirs, page, 8, "zero_dir").as_markup())

@router.callback_query(F.data.startswith("zero_dir:pick:"))
async def zero_dir_pick(c: CallbackQuery, state: FSMContext, bot: Bot):
    await c.answer()
    direction = c.data.split(":", 2)[2]
    await state.update_data(direction=direction)

    tours = get_tours_for_direction(direction)
    if not tours:
        await c.message.answer("По этому направлению в базе нет туров. Выберите другое.", reply_markup=KB_GLOBAL)
        return

    profile = build_zero_profile_text(await state.get_data())
    ranked = rank_tours(profile, tours)
    order = [i for i, _ in ranked] if ranked else list(range(len(tours)))

    await state.update_data(tours=tours, tour_order=order, tour_offset=0, chosen_tour=None)
    await push_state(state, ZeroFlow.show_tours)
    await show_next_tours(bot, c.message.chat.id, state)

async def show_next_tours(bot: Bot, chat_id: int, state: FSMContext):
    data = await state.get_data()
    tours = data.get("tours", [])
    order = data.get("tour_order", [])
    offset = int(data.get("tour_offset", 0))

    if not tours or not order:
        await bot.send_message(chat_id, "Туры не найдены. Попробуйте выбрать другое направление.", reply_markup=KB_GLOBAL)
        return

    chunk = order[offset:offset+2]
    if not chunk:
        await bot.send_message(chat_id, "Больше вариантов нет. Подтвердите анкету или измените параметры.", reply_markup=KB_GLOBAL)
        await push_state(state, ZeroFlow.summary)
        await render_zero_summary_to_chat(bot, chat_id, state)
        return

    for pos, tour_idx in enumerate(chunk, start=1):
        t = tours[tour_idx]
        url = t.get("URL", "")
        desc = t.get("описание тура", "") or "Описание скоро будет добавлено."
        n_global = offset + pos

        caption = f"Вариант тура #{n_global}\n\n{desc}"
        kb = InlineKeyboardBuilder()
        kb.row(InlineKeyboardButton(text="Выбрать этот тур", callback_data=f"zero_tour_pick:{tour_idx}"))

        if url:
            try:
                await bot.send_photo(chat_id, photo=url, caption=caption, reply_markup=kb.as_markup())
            except Exception:
                await bot.send_message(chat_id, caption + f"\n\nФото: {url}", reply_markup=kb.as_markup())
        else:
            await bot.send_message(chat_id, caption, reply_markup=kb.as_markup())

    new_offset = offset + len(chunk)
    await state.update_data(tour_offset=new_offset)

    if new_offset < len(order):
        kb_more = InlineKeyboardBuilder()
        kb_more.row(InlineKeyboardButton(text="Показать ещё варианты", callback_data="zero_more"))
        await bot.send_message(chat_id, "Показать ещё варианты?", reply_markup=kb_more.as_markup())
    else:
        await bot.send_message(chat_id, "Если хотите — выберите один из вариантов выше.", reply_markup=KB_GLOBAL)

@router.callback_query(F.data == "zero_more")
async def zero_more(c: CallbackQuery, state: FSMContext, bot: Bot):
    await c.answer()
    await show_next_tours(bot, c.message.chat.id, state)

@router.callback_query(F.data.startswith("zero_tour_pick:"))
async def zero_pick_tour(c: CallbackQuery, state: FSMContext, bot: Bot):
    await c.answer()
    tour_idx = int(c.data.split(":")[1])
    data = await state.get_data()
    tours = data.get("tours", [])
    if not tours or tour_idx >= len(tours):
        await c.message.answer("Не получилось выбрать тур. Попробуйте снова.", reply_markup=KB_GLOBAL)
        return
    await state.update_data(chosen_tour=tours[tour_idx])
    await push_state(state, ZeroFlow.summary)
    await render_zero_summary_to_chat(bot, c.message.chat.id, state)

async def render_zero_summary_to_chat(bot: Bot, chat_id: int, state: FSMContext):
    data = await state.get_data()
    txt = [
        "📌 Анкета:",
        f"Направление: {data.get('direction')}",
        f"Состав: взрослые {data.get('adults')}, дети {data.get('kids')}" + (f" (возраст: {data.get('kids_ages')})" if data.get("kids_ages") else ""),
        f"Звёзды отеля: {data.get('stars')}",
        f"Даты: {data.get('dates')}",
        f"Бюджет: {data.get('budget')}",
        f"Тип отдыха: {data.get('rest_type')}",
        "",
    ]
    chosen = data.get("chosen_tour")
    if chosen:
        txt += [
            "🎯 Подобранный тур:",
            f"Описание: {chosen.get('описание тура','') or '—'}",
            f"Картинка: {chosen.get('URL','') or '—'}",
            "",
        ]
    txt.append("Подтвердить анкету или изменить?")

    kb = InlineKeyboardBuilder()
    kb.row(InlineKeyboardButton(text="✅ Подтвердить анкету", callback_data="zero_confirm"))
    kb.row(InlineKeyboardButton(text="✏️ Изменить", callback_data="zero_edit"))
    await bot.send_message(chat_id, "\n".join(txt), reply_markup=kb.as_markup())

@router.callback_query(F.data == "zero_edit")
async def zero_edit(c: CallbackQuery, state: FSMContext):
    await c.answer()
    await push_state(state, ZeroFlow.edit_field)
    kb = InlineKeyboardBuilder()
    kb.row(InlineKeyboardButton(text="Направление", callback_data="zero_edit_field:direction"))
    kb.row(InlineKeyboardButton(text="Состав", callback_data="zero_edit_field:party"))
    kb.row(InlineKeyboardButton(text="Звёзды отеля", callback_data="zero_edit_field:stars"))
    kb.row(InlineKeyboardButton(text="Даты", callback_data="zero_edit_field:dates"))
    kb.row(InlineKeyboardButton(text="Бюджет", callback_data="zero_edit_field:budget"))
    kb.row(InlineKeyboardButton(text="Тип отдыха", callback_data="zero_edit_field:rest_type"))
    kb.row(InlineKeyboardButton(text="Тур", callback_data="zero_edit_field:tour"))
    await c.message.answer("Что изменить?", reply_markup=kb.as_markup())

@router.callback_query(F.data.startswith("zero_edit_field:"))
async def zero_edit_field_pick(c: CallbackQuery, state: FSMContext, bot: Bot):
    await c.answer()
    field = c.data.split(":")[1]

    if field == "direction":
        await state.update_data(dir_page=0, chosen_tour=None, tour_offset=0)
        await state.set_state(ZeroFlow.choose_direction)
        dirs = get_available_directions()
        await c.message.answer("Выберите направление из базы:", reply_markup=KB_GLOBAL)
        await c.message.answer("Направления:", reply_markup=paginate_buttons(dirs, 0, 8, "zero_dir").as_markup())
        return

    if field == "party":
        await state.set_state(ZeroFlow.adults)
        await c.message.answer("Сколько взрослых?", reply_markup=KB_GLOBAL)
        await c.message.answer("Выберите:", reply_markup=adults_inline("zero_adults").as_markup())
        return

    if field == "stars":
        await state.set_state(ZeroFlow.stars)
        await c.message.answer("Звёзды отеля:", reply_markup=KB_GLOBAL)
        await c.message.answer("Выберите:", reply_markup=stars_inline("zero_stars").as_markup())
        return

    if field == "dates":
        await state.set_state(ZeroFlow.dates_mode)
        await c.message.answer("Даты поездки:", reply_markup=KB_GLOBAL)
        await c.message.answer("Выберите:", reply_markup=dates_mode_inline("zero_dates").as_markup())
        return

    if field == "budget":
        await state.set_state(ZeroFlow.budget)
        await c.message.answer("Бюджет:", reply_markup=KB_GLOBAL)
        await c.message.answer("Выберите:", reply_markup=budget_inline("zero_budget").as_markup())
        return

    if field == "rest_type":
        await state.set_state(ZeroFlow.rest_type)
        await c.message.answer("Тип отдыха:", reply_markup=KB_GLOBAL)
        await c.message.answer("Выберите:", reply_markup=rest_type_inline("zero_type").as_markup())
        return

    if field == "tour":
        data = await state.get_data()
        direction = data.get("direction")
        tours = get_tours_for_direction(direction)
        profile = build_zero_profile_text(data)
        ranked = rank_tours(profile, tours)
        order = [i for i, _ in ranked] if ranked else list(range(len(tours)))
        await state.update_data(tours=tours, tour_order=order, tour_offset=0, chosen_tour=None)
        await state.set_state(ZeroFlow.show_tours)
        await show_next_tours(bot, c.message.chat.id, state)
        return

@router.callback_query(F.data == "zero_confirm")
async def zero_confirm(c: CallbackQuery, state: FSMContext):
    await c.answer()
    await push_state(state, ZeroFlow.contact_phone)
    await c.message.answer("Отправьте номер телефона (кнопкой) или напишите вручную:", reply_markup=kb_contact_request())

@router.message(ZeroFlow.contact_phone)
async def zero_phone(m: Message, state: FSMContext):
    phone = None
    if m.contact and m.contact.phone_number:
        phone = normalize_phone(m.contact.phone_number)
    else:
        phone = normalize_phone(m.text or "")
    if not phone:
        await m.answer("Не распознал телефон. Отправьте кнопкой или напишите вручную (минимум 10 цифр).", reply_markup=kb_contact_request())
        return
    await state.update_data(phone=phone)
    await push_state(state, ZeroFlow.contact_name)
    await m.answer("Введите Фамилию Имя:", reply_markup=KB_GLOBAL)

@router.message(ZeroFlow.contact_name)
async def zero_name(m: Message, state: FSMContext):
    fio = safe_text(m.text)
    if len(fio.split()) < 2:
        await m.answer("Нужно в формате «Фамилия Имя». Введите ещё раз:", reply_markup=KB_GLOBAL)
        return
    data = await state.get_data()
    phone = data.get("phone", "")
    await m.answer(
        f"Спасибо, {fio}! Мы отправили вашу заявку менеджеру — скоро с вами свяжутся для оплаты.\nТелефон: {phone}",
        reply_markup=KB_START
    )
    await reset_to_start(state)

# -------------------------
# 16) TTS callbacks (idea branch messages)
# -------------------------
@router.callback_query(F.data.startswith("tts:"))
async def tts_cb(c: CallbackQuery, bot: Bot):
    await c.answer()
    key = c.data.split(":", 1)[1]
    payload = TTS_STORE.get(key)
    if not payload:
        await c.message.answer("Не нашёл текст для озвучки (сообщение слишком старое).")
        return
    text = payload.get("text", "")
    try:
        mp3 = await openai_tts_mp3_bytes(text)
        from aiogram.types import BufferedInputFile
        audio = BufferedInputFile(mp3, filename="tts.mp3")
        await bot.send_audio(c.message.chat.id, audio=audio, caption="🔊 Озвучка (голос ИИ)")
    except Exception as e:
        await c.message.answer(f"Ошибка озвучки: {e}")

# -------------------------
# 17) Fallback
# -------------------------
@router.message()
async def fallback(m: Message, state: FSMContext):
    st = await state.get_state()
    if not st:
        await m.answer("Нажмите /start", reply_markup=KB_START)
        return
    await m.answer("Пожалуйста, используйте кнопки или ответьте на вопрос. Нажмите «Назад», если нужно.", reply_markup=KB_GLOBAL)

# -------------------------
# 18) Run bot
# -------------------------
async def main():
    bot = Bot(TELEGRAM_BOT_TOKEN)
    dp = Dispatcher(storage=MemoryStorage())
    dp.include_router(router)

    try:
        from aiogram.types import BotCommand
        await bot.set_my_commands([
            BotCommand(command="start", description="Начать заново"),
            BotCommand(command="help", description="Помощь"),
            BotCommand(command="contacts", description="Контакты менеджера"),
            BotCommand(command="site", description="Сайт"),
        ])
    except Exception:
        pass

    try:
        await bot.delete_webhook(drop_pending_updates=True)
    except Exception:
        pass

    await dp.start_polling(bot, allowed_updates=dp.resolve_used_update_types())

asyncio.run(main())


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 698.4/698.4 kB 9.1 MB/s eta 0:00:00
